# Notebook 02 — Environmental Datasets: Zimbabwe's Climate in Context

**GeoAI Workshop — Climate, Environment and Health in Zimbabwe**  
25 June 2026 · YSN Bulawayo

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Load and explore monthly climate data derived from satellite products (CHIRPS rainfall, MODIS LST, NDVI).
2. Describe Zimbabwe's seasonal cycle: rainy season October–April, dry season May–September.
3. Identify the 2015 El Niño drought signal in rainfall deficits and temperature anomalies.
4. Visualise spatial gradients: south/southwest Zimbabwe is hotter and drier; the Eastern Highlands are cooler and wetter.
5. Connect satellite-derived products to health-relevant gestational exposures.

---

**Dataset:** Monthly climate time series for 399 DHS clusters, 2011–2015 (23,940 cluster-month records).  
Variables are derived from: CHIRPS v2.0 (rainfall), MODIS MOD11A2 (land surface temperature), MODIS MOD13A3 (NDVI).

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.cm import get_cmap

DATA_DIR = pathlib.Path().resolve().parent / 'data'

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

---
## 1. Load Climate Data and Merge with Cluster Metadata

The climate dataset contains one row per cluster per month. We merge it with the cluster summary table to gain province labels and coordinates for spatial plots.

In [ ]:
climate = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters = pd.read_csv(DATA_DIR / 'zw_clusters.csv')

# Merge province and coordinates onto the climate records
climate = climate.merge(
    clusters[['cluster_id', 'province', 'cluster_lat', 'cluster_lon', 'urban']],
    on='cluster_id', how='left'
)

print(f"Climate records: {len(climate):,}")
print(f"Clusters: {climate['cluster_id'].nunique()}  |  Years: {climate['year'].min()}–{climate['year'].max()}")
print(f"Months per cluster-year: {len(climate) / climate['cluster_id'].nunique() / climate['year'].nunique():.0f}")
print()
print("Variable ranges:")
print(f"  lst_day_c : {climate['lst_day_c'].min():.1f} – {climate['lst_day_c'].max():.1f} °C")
print(f"  chirps_mm : {climate['chirps_mm'].min():.1f} – {climate['chirps_mm'].max():.1f} mm/month")
print(f"  ndvi      : {climate['ndvi'].min():.3f} – {climate['ndvi'].max():.3f}")

In [ ]:
climate.head(4)

**Product notes:**
- **CHIRPS** (Climate Hazards Group InfraRed Precipitation with Station data): ~5 km gridded monthly rainfall combining cold cloud duration from geostationary satellites with rain-gauge station data. Values are millimetres per month.
- **MODIS LST** (MOD11A2): 1 km daytime land surface temperature from the Terra satellite, 8-day composites. `lst_day_c` here is the monthly mean of available 8-day composites. LST is not air temperature — it can be 10–15 °C hotter than 2m air temperature over bare soil in the dry season.
- **MODIS NDVI** (MOD13A3): 1 km monthly NDVI. Values range from −1 to 1; cropland and woodland in Zimbabwe typically fall in 0.1–0.7 during the growing season, dropping below 0.2 in the dry season.

---
## 2. Zimbabwe's Seasonal Cycle

Zimbabwe has a subtropical climate with a single dominant rainy season driven by the Intertropical Convergence Zone (ITCZ). The country is in the austral summer monsoon belt:

- **Rainy season:** October–April (peaking December–February)
- **Dry season:** May–September

Temperatures follow a bimodal pattern: the **hot-dry season** (October–November) sees the highest daytime LST, just before the rains arrive and cool the land surface. This is the period of peak heat exposure for first-trimester pregnancies conceived in July–August.

We compute the baseline seasonal cycle using 2011–2014 (four full pre-El-Niño years), averaging over all clusters.

In [ ]:
baseline = climate[climate['year'].isin([2011, 2012, 2013, 2014])]

# National mean by calendar month, averaging over all clusters and baseline years
seasonal = baseline.groupby('month')[['lst_day_c', 'chirps_mm', 'ndvi']].mean()

fig, ax1 = plt.subplots(figsize=(10, 5))

# Shade the rainy season (Oct=10 through Apr=4, wrapping the year)
rainy_months = [10, 11, 12, 1, 2, 3, 4]
for m in rainy_months:
    ax1.axvspan(m - 0.5, m + 0.5, color='#d6eaf8', alpha=0.55, zorder=0)

ax2 = ax1.twinx()

l1, = ax1.plot(seasonal.index, seasonal['lst_day_c'], color='#c0392b',
               marker='o', markersize=5, linewidth=2, label='LST daytime (°C)')
l2, = ax2.bar(seasonal.index, seasonal['chirps_mm'], color='#2980b9',
              alpha=0.65, width=0.65, label='Rainfall (mm/month)') if False else (None,)
ax2.bar(seasonal.index, seasonal['chirps_mm'], color='#2980b9', alpha=0.65,
        width=0.65, zorder=2)

ax1.set_xlabel('Month')
ax1.set_ylabel('Mean daytime LST (°C)', color='#c0392b')
ax2.set_ylabel('Mean rainfall (mm/month)', color='#2980b9')
ax1.tick_params(axis='y', labelcolor='#c0392b')
ax2.tick_params(axis='y', labelcolor='#2980b9')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

rainy_patch = mpatches.Patch(color='#d6eaf8', label='Rainy season (Oct–Apr)')
lst_line = mpatches.Patch(color='#c0392b', label='Daytime LST (°C)')
rain_bar = mpatches.Patch(color='#2980b9', alpha=0.65, label='Rainfall (mm/month)')
ax1.legend(handles=[rainy_patch, lst_line, rain_bar], loc='upper left', fontsize=9)

ax1.set_title('Zimbabwe Seasonal Climate Cycle — Baseline 2011–2014\n'
              'Mean over 399 DHS clusters; blue shading = rainy season')
plt.tight_layout()
plt.show()

The chart confirms the expected pattern: LST peaks in October–November (hot-dry season) and drops once the rains begin in November–December. Rainfall is nearly absent May–September. For pregnancies conceived in the dry season (May–September), the third trimester (which determines final fetal growth velocity) falls during the main rains — rainfall and food availability are higher, but heat exposure in the first trimester at conception was near its annual peak.

---
## 3. Exercise 1 — The 2015 El Niño Rainfall Deficit (3 points)

The 2015–16 El Niño was one of the strongest on record and caused severe drought across southern Africa. Zimbabwe's 2015/16 growing season (Oct 2015–Mar 2016) received substantially below-average rainfall, with cascading impacts on food security and, hypothetically, on maternal nutrition and birth outcomes.

**Task:**
1. Compute the national mean monthly rainfall for **2015** and for the **2011–2014 baseline**, averaging over all clusters.
2. Store as `rain_2015` and `rain_baseline` (pandas Series, index = month 1–12).
3. Plot both series on the same axes with the deficit shaded.
4. Store the total annual deficit in `annual_rain_deficit_mm` (float = `sum(baseline) − sum(2015)`).

In [ ]:
### BEGIN SOLUTION
rain_2015 = (
    climate[climate['year'] == 2015]
    .groupby('month')['chirps_mm'].mean()
)
rain_baseline = (
    climate[climate['year'].isin([2011, 2012, 2013, 2014])]
    .groupby('month')['chirps_mm'].mean()
)

annual_rain_deficit_mm = float(rain_baseline.sum() - rain_2015.sum())

fig, ax = plt.subplots(figsize=(10, 5))
months = rain_baseline.index
ax.fill_between(months,
                rain_baseline.values, rain_2015.values,
                where=(rain_baseline.values > rain_2015.values),
                color='#e74c3c', alpha=0.3, label='Rainfall deficit')
ax.plot(months, rain_baseline, color='#2980b9', linewidth=2,
        marker='o', markersize=5, label='Baseline 2011–2014')
ax.plot(months, rain_2015, color='#e74c3c', linewidth=2,
        marker='s', markersize=5, linestyle='--', label='2015 (El Niño year)')
ax.set_xlabel('Month')
ax.set_ylabel('Mean rainfall (mm/month)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_title(f'Zimbabwe Rainfall: 2015 vs 2011–2014 Baseline\n'
             f'Total annual deficit: {annual_rain_deficit_mm:.0f} mm')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Annual rainfall deficit (baseline − 2015): {annual_rain_deficit_mm:.1f} mm")
### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert 'rain_2015' in dir(), "rain_2015 not defined"
assert 'rain_baseline' in dir(), "rain_baseline not defined"
assert 'annual_rain_deficit_mm' in dir(), "annual_rain_deficit_mm not defined"
assert isinstance(rain_2015, pd.Series), "rain_2015 must be a pandas Series"
assert isinstance(rain_baseline, pd.Series), "rain_baseline must be a pandas Series"
assert isinstance(annual_rain_deficit_mm, float), "annual_rain_deficit_mm must be a float"
assert len(rain_2015) == 12, f"rain_2015 must have 12 months, got {len(rain_2015)}"
assert len(rain_baseline) == 12, f"rain_baseline must have 12 months, got {len(rain_baseline)}"
# Verify index is months 1-12
assert set(rain_2015.index) == set(range(1, 13)), "rain_2015 index must be months 1–12"
assert set(rain_baseline.index) == set(range(1, 13)), "rain_baseline index must be months 1–12"
# Verify reference values
_r2015 = climate[climate['year'] == 2015].groupby('month')['chirps_mm'].mean()
_rbase = climate[climate['year'].isin([2011,2012,2013,2014])].groupby('month')['chirps_mm'].mean()
_def = float(_rbase.sum() - _r2015.sum())
assert abs(annual_rain_deficit_mm - _def) < 1.0, \
    f"annual_rain_deficit_mm: expected {_def:.1f}, got {annual_rain_deficit_mm:.1f}"
# 2015 should be drier overall than baseline during the rainy season
for m in [11, 12, 1, 2]:
    assert rain_2015[m] <= rain_baseline[m] + 10, \
        f"Month {m}: 2015 rainfall should not greatly exceed baseline (El Niño was a drought year)"
print("Exercise 1 tests passed.")
### END HIDDEN TESTS

---
## 4. Spatial Patterns in LST and NDVI

Zimbabwe spans a significant north-south and east-west environmental gradient. The **Eastern Highlands** (Manicaland province, longitudes > 31.5°E) are cooler and wetter due to orographic lift from the Mozambique Channel moisture. The **Lowveld** in the south (Masvingo, Matabeleland South) is consistently hotter and more arid.

We compute cluster-level means over the full 2011–2015 period and map them.

In [ ]:
# Mean annual LST and NDVI for each cluster, averaged over all months and years
cluster_means = (
    climate.groupby(['cluster_id', 'cluster_lat', 'cluster_lon'])
    [['lst_day_c', 'ndvi', 'chirps_mm']]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# LST panel
sc1 = axes[0].scatter(cluster_means['cluster_lon'], cluster_means['cluster_lat'],
                      c=cluster_means['lst_day_c'], cmap='RdYlBu_r',
                      s=22, alpha=0.9, linewidths=0)
cb1 = plt.colorbar(sc1, ax=axes[0], shrink=0.75)
cb1.set_label('Mean daytime LST (°C)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Mean Daytime LST 2011–2015\nper DHS cluster')
axes[0].axvline(31.5, color='k', linestyle=':', linewidth=0.8, alpha=0.6)
axes[0].text(31.6, -18.5, 'Eastern\nHighlands', fontsize=8, color='#2c3e50')

# NDVI panel
sc2 = axes[1].scatter(cluster_means['cluster_lon'], cluster_means['cluster_lat'],
                      c=cluster_means['ndvi'], cmap='YlGn',
                      s=22, alpha=0.9, linewidths=0)
cb2 = plt.colorbar(sc2, ax=axes[1], shrink=0.75)
cb2.set_label('Mean NDVI')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('Mean NDVI 2011–2015\nper DHS cluster')
axes[1].axvline(31.5, color='k', linestyle=':', linewidth=0.8, alpha=0.6)

plt.suptitle('Spatial Gradients in Zimbabwe Climate — Annual Means 2011–2015', y=1.01)
plt.tight_layout()
plt.show()

The expected gradients are visible: clusters east of 31.5°E (Eastern Highlands) are cooler and greener. The southwestern Lowveld (Matabeleland South / southern Masvingo) shows the highest LST and lowest NDVI, consistent with a semi-arid savanna environment.

---
## 5. Exercise 2 — Hottest Province in 2015 (3 points)

**Task:** Compute the mean 2015 daytime LST for each province, averaging over all clusters assigned to that province and all 12 months of 2015. Identify which province had the highest mean LST.

**Store:**
- `lst_by_province_2015` — pandas Series or DataFrame with province as index and mean LST as values
- `hottest_province_2015` — string: name of the province with the highest mean LST in 2015

In [ ]:
### BEGIN SOLUTION
clim_2015 = climate[climate['year'] == 2015]

lst_by_province_2015 = (
    clim_2015.groupby('province')['lst_day_c']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

hottest_province_2015 = str(lst_by_province_2015.idxmax())

print("Mean daytime LST by province (2015):")
print(lst_by_province_2015.to_string())
print(f"\nHottest province: {hottest_province_2015} ({lst_by_province_2015.max():.2f} °C)")
### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert 'lst_by_province_2015' in dir(), "lst_by_province_2015 not defined"
assert 'hottest_province_2015' in dir(), "hottest_province_2015 not defined"
assert isinstance(hottest_province_2015, str), "hottest_province_2015 must be a string"
# Accept Series or single-column DataFrame
_is_series = isinstance(lst_by_province_2015, pd.Series)
_is_df = isinstance(lst_by_province_2015, pd.DataFrame)
assert _is_series or _is_df, "lst_by_province_2015 must be a Series or DataFrame"
# Verify the hottest province against reference
_ref = (
    climate[climate['year'] == 2015]
    .groupby('province')['lst_day_c'].mean()
)
assert hottest_province_2015 == _ref.idxmax(), \
    f"Expected '{_ref.idxmax()}', got '{hottest_province_2015}'"
# Verify at least some province values match reference
if _is_series:
    for prov in _ref.index[:3]:
        if prov in lst_by_province_2015.index:
            assert abs(lst_by_province_2015[prov] - _ref[prov]) < 0.1, \
                f"Province {prov}: expected {_ref[prov]:.2f}, got {lst_by_province_2015[prov]:.2f}"
assert len(_ref) == climate['province'].nunique(), "Should have one entry per province"
print("Exercise 2 tests passed.")
### END HIDDEN TESTS

---
## 6. NDVI and Vegetation Stress During El Niño

NDVI (Normalised Difference Vegetation Index) is a proxy for green biomass and canopy cover. In agricultural regions, it tracks crop development and harvest potential. A suppressed NDVI during the growing season (Oct–Apr) indicates reduced rainfall, crop stress, and lower agricultural yields — directly relevant to maternal nutrition and food security.

We compare monthly mean NDVI for 2015 against the 2011–2014 baseline.

In [ ]:
ndvi_2015 = (
    climate[climate['year'] == 2015]
    .groupby('month')['ndvi'].mean()
)
ndvi_baseline = (
    climate[climate['year'].isin([2011, 2012, 2013, 2014])]
    .groupby('month')['ndvi'].mean()
)

fig, ax = plt.subplots(figsize=(10, 5))

# Shade rainy season
for m in [10, 11, 12, 1, 2, 3, 4]:
    ax.axvspan(m - 0.5, m + 0.5, color='#d6eaf8', alpha=0.45, zorder=0)

ax.fill_between(ndvi_baseline.index,
                ndvi_baseline.values, ndvi_2015.values,
                where=(ndvi_baseline.values > ndvi_2015.values),
                color='#e67e22', alpha=0.3, label='NDVI deficit (2015 below baseline)')
ax.plot(ndvi_baseline.index, ndvi_baseline.values, color='#27ae60',
        linewidth=2, marker='o', markersize=5, label='Baseline 2011–2014')
ax.plot(ndvi_2015.index, ndvi_2015.values, color='#e67e22',
        linewidth=2, marker='s', markersize=5, linestyle='--', label='2015 (El Niño year)')

ax.set_xlabel('Month')
ax.set_ylabel('Mean NDVI (national, all clusters)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
rainy_patch = mpatches.Patch(color='#d6eaf8', label='Rainy season (Oct–Apr)')
ax.legend(handles=[rainy_patch] + ax.get_legend_handles_labels()[0], fontsize=9)
ax.set_title('NDVI Monthly Mean: 2015 vs 2011–2014 Baseline\nVegetation suppression during the El Niño growing season')
plt.tight_layout()
plt.show()

NDVI during the 2015 Oct–Dec period (the onset of the growing season) is visibly below baseline — a direct consequence of delayed and below-average rains. By the time field crops should be germinating and establishing, the El Niño drought was suppressing vegetation growth.

**Why this matters for maternal health:** Lower NDVI in the growing season is a proxy for reduced crop production and dietary diversity. Women pregnant during the 2015/16 growing season — particularly in rural, agriculturally-dependent households — may have had reduced access to fresh vegetables, legumes, and micronutrient-rich foods. This pathway from drought → food insecurity → maternal undernutrition → fetal growth restriction underpins the biological plausibility of drought effects on birth outcomes that we investigate in Notebooks 05 and 06.

---
## 7. Exercise 3 — Rainfall–NDVI Temporal Correlation (2 points)

The relationship between monthly rainfall and NDVI is not straightforward: vegetation responds to antecedent moisture with a lag. In Zimbabwe's savanna, grasses green up after the rains arrive (Nov–Dec) and remain green through autumn (Mar–May) even as rainfall declines — meaning peak NDVI occurs *after* peak rainfall. This lag effect is worth quantifying.

**Task:** Compute the Pearson correlation coefficient between monthly mean `chirps_mm` and monthly mean `ndvi`, across **all** cluster-month records in the full dataset. The direction of this instantaneous (zero-lag) correlation may be counterintuitive given the phenological lag described above.

**Store:**
- `rain_ndvi_r` — float: Pearson r, rounded to 3 decimal places
- `rain_ndvi_direction` — string: `'positive'` or `'negative'`

In [ ]:
### BEGIN SOLUTION
# Pearson r across all cluster-month records (zero-lag)
rain_ndvi_r = round(
    float(climate[['chirps_mm', 'ndvi']].corr().loc['chirps_mm', 'ndvi']),
    3
)
rain_ndvi_direction = 'positive' if rain_ndvi_r > 0 else 'negative'

print(f"Pearson r (rainfall vs NDVI): {rain_ndvi_r}")
print(f"Direction: {rain_ndvi_direction}")
print()
print("The negative sign reflects the phenological lag: NDVI peaks in the dry season")
print("(Apr–Jul) because vegetation accumulated from the preceding rains persists for")
print("months. The zero-lag correlation between concurrent rain and NDVI is therefore")
print("negative — rain is high when NDVI is just starting to respond (Nov–Dec).")
print("A lagged correlation (e.g. NDVI ~ rainfall 1–2 months prior) would be positive.")
### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert 'rain_ndvi_r' in dir(), "rain_ndvi_r not defined"
assert 'rain_ndvi_direction' in dir(), "rain_ndvi_direction not defined"
assert isinstance(rain_ndvi_r, float), "rain_ndvi_r must be a float"
assert isinstance(rain_ndvi_direction, str), "rain_ndvi_direction must be a string"
assert rain_ndvi_direction in ('positive', 'negative'), \
    "rain_ndvi_direction must be 'positive' or 'negative'"
assert -1.0 <= rain_ndvi_r <= 1.0, f"Pearson r must be in [-1, 1], got {rain_ndvi_r}"
# Verify against reference
_ref_r = round(float(climate[['chirps_mm', 'ndvi']].corr().loc['chirps_mm', 'ndvi']), 3)
assert abs(rain_ndvi_r - _ref_r) < 0.005, \
    f"rain_ndvi_r: expected {_ref_r}, got {rain_ndvi_r}. Round to 3dp."
assert rain_ndvi_direction == ('positive' if _ref_r > 0 else 'negative'), \
    f"rain_ndvi_direction should be '{'positive' if _ref_r > 0 else 'negative'}'"
print("Exercise 3 tests passed.")
### END HIDDEN TESTS

---
## Recap

**Key points from this notebook:**

- **Satellite products** (CHIRPS, MODIS LST, MODIS NDVI) provide spatially continuous, long-term climate records that are the primary data source for environmental health research in sub-Saharan Africa, where ground-station networks are sparse.

- **Zimbabwe's seasonal cycle** is dominated by a single rainy season (Oct–Apr). The hot-dry season (Oct–Nov) immediately before the rains presents the highest heat exposure — relevant for pregnancies in their first trimester at that time.

- **The 2015 El Niño** produced a marked rainfall deficit relative to the 2011–2014 baseline, particularly during the growing season onset (Oct–Dec 2015). This is visible in both rainfall totals and suppressed NDVI during what should be the growing season.

- **Spatial gradients** are strong: Eastern Highlands clusters are systematically cooler and greener; the southwestern Lowveld is hotter and more arid. These gradients create spatial confounding that must be accounted for in exposure–outcome analyses.

- **NDVI–rainfall phenological lag:** The zero-lag Pearson correlation between concurrent monthly rainfall and NDVI is negative in this dataset because NDVI peaks in the dry season, sustained by moisture from the preceding rainy season. A lagged correlation (NDVI against rainfall 1–3 months prior) recovers the expected positive relationship and is the appropriate quantity for linking rainfall to vegetation productivity.

**In Notebook 03** we perform the spatial join that links these cluster-level climate records to individual birth records by cluster ID and gestational period — producing the trimester-level exposure columns (`lst_t1/t2/t3`, `rain_t1/t2/t3`, `ndvi_t1/t2/t3`) that are already present in `zw_births.csv`.